In [ ]:
import torch
import transformers
import peft
import accelerate
import numpy as np

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("numpy:", np.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt

from torchvision.datasets import CIFAR10
from torchvision import transforms
from torch.utils.data import DataLoader, random_split, TensorDataset
from transformers import AutoModel, AutoImageProcessor
from peft import LoraConfig, get_peft_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## data

In [ ]:
# for preprocessing the image before passing into ijepa model
processor = AutoImageProcessor.from_pretrained("facebook/ijepa_vith14_1k")
# loading the pretrained backbone
backbone = AutoModel.from_pretrained(
    "facebook/ijepa_vith14_1k", 
    torch_dtype=torch.float16, # float16 for less gpu time
    attn_implementation="sdpa").to(device) # scaled dot product attention

In [ ]:
print(processor)
print(processor.to_dict())

In [ ]:
# LOADING CIFAR10 DATASET

# processing the img
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

train_full = CIFAR10(root="./data", train=True, download=True, transform=transform)
test_ds = CIFAR10(root="./data", train=False, download=True, transform=transform)

train_ds, val_ds = random_split(
    train_full,
    [45000, 5000],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True) # pin_memory true can make moving images from cpu to gpu faster
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
# batch size for train is smaller as training takes up more memory

## extract frozen i jepa features

In [ ]:
# because we dont need to train it
backbone.eval()

# freezing all parameters
for p in backbone.parameters():
    p.requires_grad = False

# gets the dim of the last layer (feature vector produced by the model)
hidden_size = backbone.config.hidden_size

@torch.no_grad()
def extract_features(loader):
    xs, ys = [], []
    for images, labels in loader:
        images = images.to(device, dtype=torch.float16)
        outputs = backbone(pixel_values=images) # shape (batch size, no. of tokens, emb dim)
        # mean pooling across token dim to get one embedding per image
        feats = outputs.last_hidden_state.mean(dim=1) # shape (batch_size, emb dim)
        # keeping the extracted features in the gpu will waste unnecessary gpu memory which is not needed
        # so shift them back to cpu and convert them back to float32
        xs.append(feats.float().cpu())
        ys.append(labels.cpu())
    return torch.cat(xs), torch.cat(ys)

X_train, y_train = extract_features(train_loader)
X_val, y_val = extract_features(val_loader)
X_test, y_test = extract_features(test_loader)

torch.save({
    "X_train": X_train, "y_train": y_train,
    "X_val": X_val, "y_val": y_val,
    "X_test": X_test, "y_test": y_test,
    "hidden_size": hidden_size,
}, "/kaggle/working/ijepa_features.pt")

print(X_train.shape, X_val.shape, X_test.shape)

## linear probe

In [ ]:
# calculating mean and std for normalizing the feature vectors
feat_mean = X_train.mean(dim=0, keepdim=True)
feat_std = X_train.std(dim=0, keepdim=True).clamp_min(1e-6) # clamp min prevents division by zero in case any feature has almost no variation

# normalizing the features
X_train_n = (X_train - feat_mean) / feat_std
# we normalize train and val features also using training mean and std to prevent data leakage
X_val_n = (X_val - feat_mean) / feat_std
X_test_n = (X_test - feat_mean) / feat_std

# maps the feature vector to 10 classes of cifar10
linear_probe = nn.Linear(hidden_size, 10).to(device)
# defining optimizer and loss function
optimizer = torch.optim.AdamW(linear_probe.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()


# wrapping the feature vectors around the TensorDataset class for convenience
train_feat_ds = TensorDataset(X_train_n, y_train)
val_feat_ds = TensorDataset(X_val_n, y_val)
test_feat_ds = TensorDataset(X_test_n, y_test)

train_feat_loader = DataLoader(train_feat_ds,batch_size=512,shuffle=True)
val_feat_loader = DataLoader(val_feat_ds,batch_size=512,shuffle=False)
test_feat_loader = DataLoader(test_feat_ds,batch_size=512,shuffle=False)


def run_probe_epoch(X, y, train=True, batch_size=512):
    linear_probe.train(train)
    total_loss, correct, total = 0, 0, 0

    for xb, yb in loader:
        # move features and label batch to gpu
        xb = xb.to(device)
        yb = yb.to(device)

        if train:
            optimizer.zero_grad()

        logits = linear_probe(xb)
        loss = criterion(logits, yb)

        if train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * len(xb) # loss.item() is avg loss for this batch
        correct += (logits.argmax(1) == yb).sum().item() # logits.argmax(1) gives predicted class index
        total += len(xb)

    return total_loss / total, correct / total

linear_history = []
best_val = 0

for epoch in range(50):
    tr_loss, tr_acc = run_probe_epoch(train_feat_loader, train=True)
    val_loss, val_acc = run_probe_epoch(val_feat_loader, train=False)

    linear_history.append({
        "epoch": epoch+1,
        "train_loss": tr_loss,
        "train_acc": tr_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

    pd.DataFrame(linear_history).to_csv("/kaggle/working/ijepa_linear_probe_history.csv", index=False)

    if val_acc > best_val:
        best_val = val_acc
        torch.save({
            "model_state_dict": linear_probe.state_dict(),
            "feature_mean": feat_mean,
            "feature_std": feat_std,
            "hidden_size": hidden_size,
            "best_val_acc": best_val,
            "history": linear_history,
        }, "/kaggle/working/ijepa_linear_probe_best.pt")

    print(f"Epoch {epoch+1:02d}: train_acc={tr_acc:.4f}, val_acc={val_acc:.4f}")

test_loss, test_acc = run_probe_epoch(X_test_n, y_test, False)
print("Linear probe test accuracy:", test_acc)

## linear curve

In [ ]:
linear_df = pd.DataFrame(linear_history)

plt.figure(figsize=(7,4))
plt.plot(linear_df["epoch"], linear_df["train_acc"], label="train")
plt.plot(linear_df["epoch"], linear_df["val_acc"], label="val")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("I-JEPA Linear Probe on CIFAR-10")
plt.legend()
plt.grid(True)
plt.savefig("/kaggle/working/ijepa_linear_probe_curve.png", dpi=150)
plt.show()

## clear memory

In [ ]:
del backbone
torch.cuda.empty_cache()

## lora

In [ ]:
# quantization means storing model weights with fewer bits than usual
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# loading the pretrained i jepa model with quantization settings - i.e loading the weights with fewer bits
backbone_lora = AutoModel.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
)

# this prepares the model for training with quantization (idk what that means)
backbone_lora = prepare_model_for_kbit_training(backbone_lora) # this line can be skipped when doing normal lora

target_modules = ["query", "key", "value"] # this tells lora where to insert the adapter layers

lora_config = LoraConfig(
    r=16, # rank
    lora_alpha=32, # controls the scaling of lora update (a common choice is generally 2 * r)
    lora_dropout=0.05, 
    bias="none", # means bias parameters are not trained, only lora adapter weights are trained
    target_modules=target_modules,
)

# peft - parameter efficient fine tuning (lora is one peft method)

# this takes the i jepa backbone and attaches lora adapter according to lora config
backbone_lora = get_peft_model(backbone_lora, lora_config)
backbone_lora.print_trainable_parameters()

## classifier + lora training

In [ ]:
# this class combines the i jepa and classifer
# i jepa gives features and classifier classifies them
class IJEPAClassifier(nn.Module):
    def __init__(self, backbone, hidden_size, num_classes=10):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values)
        x = outputs.last_hidden_state.mean(dim=1)
        return self.classifier(x.float())

lora_model = IJEPAClassifier(backbone_lora, backbone_lora.config.hidden_size)
lora_model.classifier = lora_model.classifier.to(device)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, lora_model.parameters()), # this line means only give optimizer the parameters that are trainable
    lr=1e-4,
    weight_decay=1e-4,
)

criterion = nn.CrossEntropyLoss()


def evaluate_lora(loader):
    lora_model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = lora_model(images)
            loss = criterion(logits, labels)

            total_loss += loss.item() * images.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

lora_history = []
best_val_acc = 0

for epoch in range(5):
    lora_model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = lora_model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_loss = total_loss / total
    train_acc = correct / total
    val_loss, val_acc = evaluate_lora(val_loader)

    lora_history.append({
        "epoch": epoch+1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

    pd.DataFrame(lora_history).to_csv("/kaggle/working/ijepa_lora_history.csv", index=False)

    torch.save({
        "epoch": epoch+1,
        "classifier_state_dict": lora_model.classifier.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": lora_history,
        "target_modules": target_modules,
        "model_name": model_name,
    }, "/kaggle/working/ijepa_lora_latest.pt")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        backbone_lora.save_pretrained("/kaggle/working/ijepa_lora_best_adapter")
        torch.save({
            "classifier_state_dict": lora_model.classifier.state_dict(),
            "best_val_acc": best_val_acc,
            "epoch": epoch+1,
            "model_name": model_name,
            "target_modules": target_modules,
        }, "/kaggle/working/ijepa_lora_best_classifier.pt")

    print(f"Epoch {epoch+1}: train_acc={train_acc:.4f}, val_acc={val_acc:.4f}")

test_loss_lora, test_acc_lora = evaluate_lora(test_loader)
print("LoRA test accuracy:", test_acc_lora)

In [ ]:
results = pd.DataFrame([
    {"model": "I-JEPA", "protocol": "Linear Probe", "test_accuracy": test_acc},
    {"model": "I-JEPA", "protocol": "LoRA / QLoRA Fine-tune", "test_accuracy": test_acc_lora},
])

results.to_csv("/kaggle/working/ijepa_results.csv", index=False)
results